In [508]:
import numpy as np
import random
import matplotlib.pyplot as plt
import time
import copy
import networkx as nx
import ipycytoscape

In [509]:
def load_circuit_from_file(filepath):
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    N = int(lines[0].strip())

    s, t, E = lines[1].split()
    s = int(s)
    t = int(t)
    E = float(E)
    CG = nx.Graph()
    for i in range(N):
        CG.add_node(i, point=str(i))
    for v, u, r in [tuple(l.split()) for l in lines[2:]]:
        CG.add_edge(int(v), int(u), R=float(r), G=1/float(r), I = 0.0)

    return CG, s, t, E

def show_circuit(CG):
    cytoscapeobj = ipycytoscape.CytoscapeWidget()
    cytoscapeobj.graph.add_graph_from_networkx(CG)
    cytoscapeobj.set_style([
    {
        'selector': 'node',
        'css': {
            'content': 'data(point)', 
            'text-valign': 'center',
            'color': 'black',
            'background-color': 'lightgray'
        }
    },
    {
        'selector': 'edge',
        'css': {
            'content': 'data(I)', 
            'font-size': '12px',
            'text-background-color': 'white',
            'text-background-opacity': 0.8
        }
    }
    ])
    display(cytoscapeobj)

def print_matrix(X):
    for row in X:
        for x in row:
            print(round(x,2), end='\t')
        print()

In [510]:
 

def nodal_analysis(CG : nx.Graph, s : int, t : int, E : float):

    '''
    params:
    G -> networkx graph representing a circuit
    s, t -> two nodes with a voltage of E between them
    E -> voltage
    '''

    N = len(CG.nodes)
    X = np.zeros((N-2,N-2))
    Y = np.zeros(N-2)

    unk = list(filter(lambda i : i != s and i != t, range(N)))

    for i in range(N-2):
        for j in range(N-2):
            if i == j:
                X[i][i]= sum(list(map(lambda edge : CG[edge[0]][edge[1]]["G"], CG.edges(unk[i]))))
            else:
                edge_data = CG.get_edge_data(unk[i], unk[j], default=0)
                X[i][j] = 0 if edge_data == 0 else -edge_data["G"]

    for i in range(N-2):
        edge_data = CG.get_edge_data(unk[i], s, default=0)
        Y[i] = 0 if edge_data == 0 else -edge_data["G"]*E 
    
    Vunk = list(np.linalg.solve(X, Y))
    V = []
    if s < t:
        V = Vunk[0:s] + [E] + Vunk[s:t] + [0] + Vunk[t:-1]
    else:
        V = Vunk[0:t] + [0] + Vunk[t:s] + [E] + Vunk[s:-1]

    for i in range(N):
        for j in range(N):
            if (i,j) not in CG.edges: continue
            U = V[j]-V[i]
            I = float(CG[i][j]["G"] * U)
            CG[i][j]["I"] =I
            



In [511]:
CG,s,t,E = load_circuit_from_file("testgraph")
nodal_analysis(CG,s,t,E)
show_circuit(CG)

CytoscapeWidget(cytoscape_layout={'name': 'cola'}, cytoscape_style=[{'selector': 'node', 'css': {'content': 'd…